## Range and Composite Indexes in Azure Cosmos DB

### Installing Utilities and Libraries

In [ ]:
%pip install azure-cosmos==4.16.0 azure-identity python-dotenv

### Setting up the Environment

In [ ]:
import os 
from dotenv import load_dotenv
import json

load_dotenv()

endpoint = os.getenv("COSMOSDB_ENDPOINT")
key = os.getenv("COSMOSDB_KEY")
database_name = os.getenv("DATABASE_NAME")
container_name = os.getenv("CONTAINER_NAME")

### Creating the CosmosDB Client

In [ ]:
from azure.cosmos import CosmosClient

client = CosmosClient(endpoint, key)

### Navigate the Resource Hierarchy

In [ ]:
database = client.get_database_client(database_name)
container = database.get_container_client(container_name)

### Create a Query Statistics Helper Function

In [ ]:
def run_query(query):

    items = list(
        container.query_items(
            query=query,
            enable_cross_partition_query=True
        )
    )

    headers = container.client_connection.last_response_headers

    print("\n=== Query Statistics ===")
    print(f"Documents Returned : {len(items)}")
    print(f"RU Charge          : {headers.get('x-ms-request-charge')}")
    print(f"Duration (ms)      : {headers.get('x-ms-request-duration-ms')}")
    print("Query Results:")
    for item in items:
        print(f"  - {item}")

    return items

### Create a Range Index on the Rating Property

In [ ]:
container_properties = container.read()

new_index_definition = """ {
    "indexingMode": "consistent",
    "automatic": true,
    "includedPaths": [
        {
            "path": "/rating/?"
        }
    ],
    "excludedPaths": [
        {
            "path": "/*"
        }
    ],
    "fullTextIndexes": []
} """

database.replace_container(
    container = container_properties["id"],
    partition_key = container_properties['partitionKey'],
    indexing_policy = json.loads(new_index_definition)
)

print("\n=== Indexing Policy Updated ===")

### Run a Range-Filter Query After Index Optimization

In [ ]:
query = """
SELECT *
FROM c
WHERE c.rating > 4.5
"""

results = run_query(query)

### Execute a Multi-Property Sort Query

In [ ]:
query = """
SELECT *
FROM c
ORDER BY c.rating DESC, c.reviewCount DESC
"""

results = run_query(query)

### Create a Composite Index

In [ ]:
container_properties = container.read()

new_index_definition = """ {
    "indexingMode": "consistent",
    "automatic": true,
    "includedPaths": [
        {
            "path": "/rating/?"
        }
    ],
    "excludedPaths": [
        {
            "path": "/*"
        }
    ],
    "compositeIndexes": [
        [
            {
                "path": "/rating",
                "order": "descending"
            },
            {
                "path": "/reviewCount",
                "order": "descending"
            }
        ]
    ],
    "fullTextIndexes": []
} """

database.replace_container(
    container = container_properties["id"],
    partition_key = container_properties['partitionKey'],
    indexing_policy = json.loads(new_index_definition)
)

print("\n=== Indexing Policy Updated ===")

### Re-run the Multi-Property Sort Query After Composite Index Creation

In [ ]:
query = """
SELECT *
FROM c
ORDER BY c.rating DESC, c.reviewCount DESC
"""

results = run_query(query)

### Create a Filter + Sort Composite Index

In [ ]:
container_properties = container.read()

new_index_definition = """ {
    "indexingMode": "consistent",
    "automatic": true,
    "includedPaths": [
        {
            "path": "/rating/?"
        }
    ],
    "excludedPaths": [
        {
            "path": "/*"
        }
    ],
    "compositeIndexes": [
        [
            {
                "path": "/category",
                "order": "ascending"
            },
            {
                "path": "/rating",
                "order": "descending"
            }
        ]
    ],
    "fullTextIndexes": []
} """

database.replace_container(
    container = container_properties["id"],
    partition_key = container_properties['partitionKey'],
    indexing_policy = json.loads(new_index_definition)
)

print("\n=== Indexing Policy Updated ===")

### Run a Filter + Sort Query After Composite Index Creation

In [ ]:
query = """
SELECT *
FROM c
WHERE c.category = "Smoothies"
ORDER BY c.rating DESC
"""

results = run_query(query)

### Revert to the Default Indexing Policy

In [ ]:
container_properties = container.read()

new_index_definition = """ {
    "indexingMode": "consistent",
    "automatic": true,
    "includedPaths": [
        {
            "path": "/*"
        }
    ],
    "excludedPaths": [
        {
            "path": "/_etag/?"
        }
    ],
    "fullTextIndexes": []
} """

database.replace_container(
    container = container_properties["id"],
    partition_key = container_properties['partitionKey'],
    indexing_policy = json.loads(new_index_definition)
)

print("\n=== Indexing Policy Updated ===")